# 离群点分析

In [ ]:
import pandas as pd
import csv
def parse_log_line(line):
    parts = line.strip().split()
    try:
        operation = parts[0]
        p_has_id = lambda x: x != "ZRAM_W_BEGIN"
        pid = int(parts[1]) if p_has_id(operation) else None
        req_id = parts[2] if p_has_id(operation) else None
        mono_time = int(parts[-1])
        ret = (operation, pid, req_id, mono_time)
    except Exception as e:
        ret = (None, None, None, None)
    return ret

data = {}
buffered_begin = None
buffered_end = None
previous_operation = None
previous_id = None

is_kcomp = False
step_1_name = 'ZRAM_W_BEGIN'
step_6_name = 'ZRAM_W_END'
if is_kcomp:
    step_2_name = 'KCOMP_BEGIN'
    step_3_name = 'KCOMP_BEGIN'
    step_4_name = 'KCOMP_END'
    step_5_name = 'KCOMP_END'
else:
    step_2_name = 'CPFD_KWAKEUP'
    step_3_name = 'CPFD_UWAKEN'
    step_4_name = 'CPFD_UWAKEUP'
    step_5_name = 'CPFD_KWAKEN'
step_names = [step_1_name, step_2_name, step_3_name, step_4_name, step_5_name, step_6_name]
klog_names = [step_1_name, step_2_name, step_5_name, step_6_name]
ulog_names = [step_3_name, step_4_name]

with open("/home/lixr/aosp_host_working_dir/2023_06_16_17_52_24_p08_wakeup_time_CNTVCT_EL0_After_Resched_SecondTime/klog.log", "r") as kernel_log:
    for line in kernel_log:
        operation, pid, req_id, mono_time = parse_log_line(line)
        if operation not in klog_names:
            continue
        # print(line)
        # print(operation, pid, req_id, mono_time)
        if operation == 'ZRAM_W_BEGIN':
            buffered_begin = [mono_time]
            previous_operation = operation
        elif operation == step_2_name: 
            # print(address, buffered_begin, cpu_cycle, mono_time, real_time)
            data[(pid, req_id)] = buffered_begin + [mono_time]
            previous_operation = operation
        elif operation == 'ZRAM_W_END':
            if previous_operation == step_5_name:
                data[(pid, req_id)] += [mono_time]
                previous_operation = operation
        else:
            data[(pid, req_id)] += [mono_time]
            previous_operation = operation
print(len(data))


In [ ]:


with open("/home/lixr/aosp_host_working_dir/2023_06_16_17_52_24_p08_wakeup_time_CNTVCT_EL0_After_Resched_SecondTime/5833-stress_comp-753452-35984332-dbgmm.log", "r") as user_log:
    for line in user_log:
        try:
            operation, pid, req_id, mono_time = parse_log_line(line)
            if not operation in ulog_names:
                continue
            if (pid, req_id) not in data:
                continue
            data[(pid, req_id)] += [mono_time]
            if len(data[(pid, req_id)]) > 8:
                raise Exception(f"len(data[(pid, req_id)]) > 8: {len(data[(pid, req_id)])}")
        except Exception as e:
            print(line)
            print(e)
            raise e


df = pd.DataFrame.from_dict(data, orient='index', columns=[
    'step1_mono_time',
    'step2_mono_time',
    'step5_mono_time',
    'step6_mono_time',
    'step3_mono_time',
    'step4_mono_time',
])

# 然后我们调整列的顺序，按照实际步骤顺序
df = df[[
    'step1_mono_time', 
    'step2_mono_time', 
    'step3_mono_time', 
    'step4_mono_time', 
    'step5_mono_time', 
    'step6_mono_time', 
]]
# 重命名索引为'id'
# df.index.name = 'id'
# Extend one column named id to the first column

#   df[f'step{i}_real_time'] = df[f'step{i}_real_time'].astype(int)
#   df[f'step{i}_cpu_cycle'] = df[f'step{i}_cpu_cycle'].astype(int)
# 使用to_csv函数保存到output.csv，index=True表示保留索引列
# df.to_csv("output.csv", index=True)

In [ ]:
df = df.dropna(axis=0, how='any')

In [ ]:

is_inc = (df[f'step{2}_mono_time'] >= df[f'step{1}_mono_time'])
# print(len(is_inc))
# print(is_inc)
# print number of is_inc == True
# print(len(is_inc[is_inc == True]))
orig_len = len(df)
for i in range(3, 6):
  is_inc = is_inc & (df[f'step{i}_mono_time'] >= df[f'step{i-1}_mono_time'])
  # print(len(is_inc[is_inc == True]))
# print(len(is_inc))
# print(len(df))
orig_df = df.copy()
df = df[is_inc == True]
print(f"{orig_len} -> {len(df)}")

In [ ]:
orig_df[is_inc == False].astype(int)

In [ ]:
1698576513056 - 1698576503942
1698576503942 - 1698576513056

In [ ]:
import pandas as pd

# 从合并的CSV文件读取数据
# data = pd.read_csv('output.csv')

# 提取CPU cycles，mono time和real time的数据
# cpu_cycles = df[['id', 'address', 'step1_cpu_cycle', 'step2_cpu_cycle', 'step3_cpu_cycle', 'step4_cpu_cycle', 'step5_cpu_cycle', 'step6_cpu_cycle']]
# mono_time = df[['id', 'address', 'step1_mono_time', 'step2_mono_time', 'step3_mono_time', 'step4_mono_time', 'step5_mono_time', 'step6_mono_time']]
mono_time = df
# real_time = df[['id', 'address', 'step1_real_time', 'step2_real_time', 'step3_real_time', 'step4_real_time', 'step5_real_time', 'step6_real_time']]

# 将三个数据集写入到各自的CSV文件
# cpu_cycles.to_csv('cpu_cycles.csv', index=False)
# mono_time.to_csv('mono_time.csv', index=False)
# real_time.to_csv('real_time.csv', index=False)

In [ ]:

# dataframes = {'cpu_cycle': cpu_cycles, 'mono_time': mono_time, 'real_time': real_time}
# dataframes = {'cpu_cycle': cpu_cycles, 'mono_time': mono_time, 'real_time': real_time}
dataframes = {'mono_time': mono_time}

# Create new dataframes for the durations
# cpu_cycles_duration = cpu_cycles[['id', 'address']].copy()
mono_time_duration = pd.DataFrame()
# real_time_duration = real_time[['id', 'address']].copy()

# durations = {'cpu_cycle': cpu_cycles_duration, 'mono_time': mono_time_duration, 'real_time': real_time_duration}
durations = {'mono_time': mono_time_duration}

# Compute the duration for each step
for key in dataframes:
    df = dataframes[key]
    duration_df = durations[key]
    for i in range(1, 6):
        duration_df[f'step{i}_duration'] = df[f'step{i+1}_{key}'] - df[f'step{i}_{key}']
        # assert(all(duration_df[f'step{i}_duration'] > 0))
    # Save to csv
    # duration_df.to_csv(f'{key}_duration.csv', index=False)

avg_sum = 0
# 计算每个区间步骤的平均耗时
for key in dataframes:
    print(key)
    df = durations[key]
    for i in range(1, 6):
        # 根据步骤列进行分组，并计算平均值
        avg_duration = df[f'step{i}_duration'].mean()
        print(f"{step_names[i-1]} -> {step_names[i]}: " + str(float(avg_duration)))
        avg_sum += avg_duration
print(avg_sum)
        


In [ ]:
# save mono_time_duration to csv
# mono_time_duration.to_csv('mono_time_duration.csv', index=False)
import numpy as np
def is_outlier(points, thresh=3.5):
    """
    Returns a boolean array with True if points are outliers and False 
    otherwise.

    Parameters:
    -----------
        points : An numobservations by numdimensions array of observations
        thresh : The modified z-score to use as a threshold. Observations with
            a modified z-score (based on the median absolute deviation) greater
            than this value will be classified as outliers.

    Returns:
    --------
        mask : A numobservations-length boolean array.

    References:
    ----------
        Boris Iglewicz and David Hoaglin (1993), "Volume 16: How to Detect and
        Handle Outliers", The ASQC Basic References in Quality Control:
        Statistical Techniques, Edward F. Mykytka, Ph.D., Editor. 
    """
    if len(points.shape) == 1:
        points = np.array(points)[:,None]
    median = np.median(points, axis=0)
    diff = np.sum((points - median)**2, axis=-1)
    diff = np.sqrt(diff)
    med_abs_deviation = np.median(diff)
    if med_abs_deviation == 0:
        return np.array([False] * len(points))
    modified_z_score = 0.6745 * diff / med_abs_deviation

    return modified_z_score > thresh

from tabulate import tabulate

import matplotlib.pyplot as plt
import matplotlib
font = {'family' : 'SimHei',
        'weight' : 'bold',
        'size'   : 30 }
import seaborn as sns
colors = sns.color_palette("bright", 5)


titles = ["1. 内核准备", "2. 调度：内核->用户", "3. 用户压缩", "4. 调度：用户->内核", "5. 内核结束"]
matplotlib.rc('font', **font)
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set(font='SimHei')
plt.figure(figsize=(30, 30))
for step in range(1, 6):
    x = mono_time_duration[f"step{step}_duration"]
    # title = f"{step_names[step-1]} -> {step_names[step]}"
    title = titles[step-1]
    print(title)
    # plt.title(title)
    print(x.describe(percentiles=[0.25, 0.50, 0.75, 0.99, 0.999]).astype(int))
    is_filter_out = x > x.quantile(0.95)
    # is_filter_out = is_outlier(x)
    # print(tabulate(mono_time_duration[is_filter_out].head(10)))
    filtered = x[~is_filter_out]
    # filtered.hist(bins=1000)
    # sns.histplot(filtered, label = title, kde=True, color = colors[step-1], bins=100)
    sns.kdeplot(filtered, label = title, color = colors[step-1])
    print("Removed outliers: ")
    print(filtered.describe(percentiles=[0.25, 0.50, 0.75, 0.999]).astype(int))
plt.legend()
    


# for step in range(1, 8):
#   plt.figure(figsize=(40, 5))
#   x = mono_time_duration[f"step{step}_duration"]
#   title = f"{step_names[step-1]} -> {step_names[step]}"
#   plt.title(title)
#   print(title)
#   print(x.describe(percentiles=[0.25, 0.50, 0.75, 0.99]).astype(int))
#   # filter the 99% percentile points
#   filtered = x[x < x.quantile(0.99)]
#   filtered.hist(bins=40000)


In [ ]:

mono_time_duration.loc[[(29389, '979')]]

In [ ]:
mono_time.loc[[(29389, '979')]]

In [ ]:

print(matplotlib.matplotlib_fname())


In [ ]:

colors =sns.color_palette("bright", 5)

In [ ]:
print(colors)

In [ ]:
sns.palplot(colors)